<a href="https://colab.research.google.com/github/Rakshitkumar1/DCT/blob/main/mdct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

def sine_window(N):
  n=np.arange(2*N)
  return np.sin(np.pi*(n+0.5)/(2*N))

In [2]:
def mdct_matrix(N):
  n=np.arange(2*N)
  k=np.arange(N)
  return np.cos(np.pi/N*(n[None,:]+0.5+N/2)*(k[:,None]+0.5))


In [3]:
def mdct_matrix(N):
  n=np.arange(2*N)
  k=np.arange(N)
  return np.cos(np.pi/N*(n[None,:]+0.5+N/2)*(k[:,None]+0.5))

In [4]:
def mdct_encode(x,N=1024):
  L=len(x)
  num_frames=int(np.ceil(L/N))+1
  padded=np.concatenate([np.zeros(N),x,np.zeros(num_frames*N-L)])
  w,M=sine_window(N),mdct_matrix(N)
  coeffs=np.empty((num_frames,N))
  for i in range(num_frames):
    frame=padded[i*N:i*N+2*N]*w
    coeffs[i]=M@frame
  return coeffs,L

In [5]:
def mdct_decode(coeffs,L,N=1024):
  num_frames=coeffs.shape[0]
  w,M=sine_window(N),mdct_matrix(N)
  out=np.zeros((num_frames+1)*N)
  for i in range(num_frames):
    frame=(2.0/N)*(M.T@coeffs[i])*w
    out[i*N:i*N+2*N]+=frame
  return out[N:N+L]



In [6]:
if __name__=="__main__":
  fs=44100
  t=np.arange(fs*2)/fs
  x=0.6*np.sin(2*np.pi*440*t)+0.3*np.sin(2*np.pi*13200*t)
  x+=0.02*np.random.default_rng(0).standard_normal(len(x))



In [8]:
coeffs,L=mdct_encode(x,N=1024)
y=mdct_decode(coeffs,L,N=1024)
print("frames:",coeffs.shape[0],"coefficients per frame:",coeffs.shape[1])
print("max abs error:",np.max(np.abs(x-y)))
assert np.allclose(x,y,atol=1e-9),"perfect reconstruction Failed"
print("perfect reconstruction succeeded")

frames: 88 coefficients per frame: 1024
max abs error: 4.599653991022024e-13
perfect reconstruction succeeded
